In [6]:
import json
import os
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import pandas as pd
import requests
import psycopg2
from sqlalchemy import create_engine, text

In [ ]:
# Load environment variables from docker .env files via locate repo root
def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "apis").exists() and (candidate / "n8n").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory.")


def read_env_file(path: Path) -> dict:
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip()
    return values


REPO_ROOT = find_repo_root()
N8N_ENV_PATH = REPO_ROOT / "n8n" / ".env"
N8N_ENV = read_env_file(N8N_ENV_PATH)

DB_HOST = os.getenv("MODEL_DB_HOST", "localhost")
DB_PORT = int(os.getenv("MODEL_DB_PORT", "5432"))
DB_NAME = os.getenv("MODEL_DB_NAME", N8N_ENV.get("MODEL_DB_NAME", "model_db"))
DB_USER = os.getenv("MODEL_DB_USER", N8N_ENV.get("MODEL_DB_USER", "model_db_user"))
DB_PASSWORD = os.getenv("MODEL_DB_PASSWORD", N8N_ENV.get("MODEL_DB_PASSWORD", ""))

TRAINER_API_BASE = os.getenv("TRAINER_API_BASE", "http://localhost:7007")
REQUEST_TIMEOUT = float(os.getenv("REQUEST_TIMEOUT_SECONDS", "30"))

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print("Repo root:", REPO_ROOT)
print("Trainer API:", TRAINER_API_BASE)
print("DB target:", f"{DB_HOST}:{DB_PORT}/{DB_NAME}", "(user:", DB_USER + ")")

Repo root: C:\Users\linki\Documents\AutoML
Trainer API: http://localhost:7007
DB target: localhost:5432/model_db (user: model_db_user)


In [ ]:
# Establish DB connection and load model specs from DB

def get_engine():
    return create_engine(DATABASE_URL, pool_pre_ping=True)


def list_models(limit: int = 20) -> pd.DataFrame:
    query = text("""
        SELECT id, name, created_at, updated_at
        FROM public.models
        ORDER BY created_at DESC
        LIMIT :limit
    """)
    with get_engine().connect() as conn:
        df = pd.read_sql(query, conn, params={"limit": limit})
    return df


def load_model_spec(model_id: str) -> tuple[dict, dict]:
    query = text("""
        SELECT id, name, nn_taxonomy, extra_meta, created_at
        FROM public.models
        WHERE id = :model_id
        LIMIT 1
    """)
    with get_engine().connect() as conn:
        df = pd.read_sql(query, conn, params={"model_id": model_id})

    if df.empty:
        raise ValueError(f"No model found for id={model_id}")

    row = df.iloc[0].to_dict()
    model_spec = row["nn_taxonomy"]
    if isinstance(model_spec, str):
        model_spec = json.loads(model_spec)

    model_info = {
        "id": str(row["id"]),
        "name": row.get("name"),
        "created_at": str(row.get("created_at")),
        "extra_meta": row.get("extra_meta"),
    }
    return model_info, model_spec


models_df = list_models(limit=25)

### Training parameter definition
By model_ID

In [12]:
# Set one of these values before running this cell.
SELECT_MODEL_ID = "36ecedfa-e352-41f2-994e-87099373f147"

if SELECT_MODEL_ID is None:
    if models_df.empty:
        raise RuntimeError("No models available in public.models")
    SELECT_MODEL_ID = str(models_df.iloc[0]["id"])

MODEL_INFO, MODEL_SPEC = load_model_spec(SELECT_MODEL_ID)

print("Selected model:", MODEL_INFO)
print("Model spec keys:", list(MODEL_SPEC.keys()))

Selected model: {'id': '36ecedfa-e352-41f2-994e-87099373f147', 'name': 'CIFAR_SimpleResNet_64_128_256_v1', 'created_at': '2026-04-18 10:40:56.862441+00:00', 'extra_meta': {'builder_notes': 'Simple ResNet-style CNN using Conv→BN→ReLU→Conv→BN residual blocks with identity addition and stage widths 64→128→256. Downsampling via stride-2 at stage transitions and final global pooling before classification.', 'selection_mode': 'synthesized'}}
Model spec keys: ['edges', 'nodes', 'input_shape', 'exit_node_id', 'entry_node_id']


In [19]:
TRAINING_PARAMETERS = {
    "loss_function": "CrossEntropyLoss",
    "optimizer": "AdamW",
    "epochs": 100,
    "dataset_name": "cifar10",
    "batch_size": 64,
    "lr": 0.01,
    "momentum": 0.9,
}

TRAIN_VIA_API = True
POLL_TIMEOUT_SECONDS = 3600
POLL_INTERVAL_SECONDS = 60

TRAIN_REQUEST = {
    "model_spec": MODEL_SPEC,
    "training_parameters": TRAINING_PARAMETERS,
}

print(json.dumps(TRAINING_PARAMETERS, indent=2))

{
  "loss_function": "CrossEntropyLoss",
  "optimizer": "AdamW",
  "epochs": 100,
  "dataset_name": "cifar10",
  "batch_size": 64,
  "lr": 0.01,
  "momentum": 0.9
}


### API Training

In [20]:
def submit_train_job(payload: dict) -> dict:
    url = f"{TRAINER_API_BASE}/submit-train-job"
    response = requests.post(url, json=payload, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def get_job_status(job_id: str) -> dict:
    url = f"{TRAINER_API_BASE}/train-jobs/{job_id}"
    response = requests.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def get_job_result(job_id: str) -> dict:
    url = f"{TRAINER_API_BASE}/train-jobs/{job_id}/result"
    response = requests.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def wait_for_job(job_id: str, timeout_seconds: float = 3600) -> dict:
    started = time.time()
    while True:
        status = get_job_status(job_id)
        state = status.get("status")
        print(f"job={job_id} status={state}")

        if state in {"succeeded", "failed", "cancelled"}:
            return status

        if (time.time() - started) > timeout_seconds:
            raise TimeoutError(f"Timed out waiting for job {job_id}")

        time.sleep(POLL_INTERVAL_SECONDS)


if TRAIN_VIA_API:
    job = submit_train_job(TRAIN_REQUEST)
    print("Submitted training job:", job)

    final_status = wait_for_job(job["job_id"], timeout_seconds=POLL_TIMEOUT_SECONDS)
    print("Final status:", final_status)

    job_result = get_job_result(job["job_id"])
    print("Result:")
    print(json.dumps(job_result, indent=2))
else:
    print("TRAIN_VIA_API is False. Run the direct fallback cell next.")

Submitted training job: {'job_id': '0c07e0fd-6100-493f-86b8-04f0b2818171', 'status': 'queued', 'status_url': '/train-jobs/0c07e0fd-6100-493f-86b8-04f0b2818171', 'result_url': '/train-jobs/0c07e0fd-6100-493f-86b8-04f0b2818171/result'}
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b2818171 status=running
job=0c07e0fd-6100-493f-86b8-04f0b28181

### Local Training
Use this when API is unavailable or when debugging training internals.

In [ ]:


MODEL_TRAINER_APP = REPO_ROOT / "apis" / "model-trainer" / "app"
if str(MODEL_TRAINER_APP) not in sys.path:
    sys.path.insert(0, str(MODEL_TRAINER_APP))

import torch
from services.train_model import train_model
from services.test_model import test_model


@dataclass
class TrainingParams:
    loss_function: str = "CrossEntropyLoss"
    optimizer: str = "SGD"
    epochs: int = 3
    dataset_name: str = "cifar10"
    batch_size: int = 32
    lr: float = 0.01
    momentum: float = 0.9

    def model_dump(self) -> dict:
        return asdict(self)


def train_test_direct(model_spec: dict, params: dict) -> dict:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tp = TrainingParams(**params)

    trained = train_model(model_spec, training_parameters=tp, torch_device=device)
    test_res = test_model(trained.torch_model, tp, trained.test_dataloader, trained.torch_device)

    return {
        "accuracy": test_res["accuracy"],
        "test_loss": test_res["test_loss"],
        "train_loss": trained.train_loss,
        "epochs": tp.epochs,
        "device": str(trained.torch_device),
    }


if not TRAIN_VIA_API:
    direct_result = train_test_direct(MODEL_SPEC, TRAINING_PARAMETERS)
    print(json.dumps(direct_result, indent=2))
else:
    print("Skip direct training because TRAIN_VIA_API=True")